In [0]:
# ============================================================
# Project  : ABC E-Commerce
# Layer    : Silver
# Notebook : 08_Silver_Orders
# Author   : Suriya
# Purpose  : Clean and validate Orders Data
# ============================================================

In [0]:
from pyspark.sql.functions import *

In [0]:
orders_df = spark.table("workspace.default.bronze_orders")

In [0]:
orders_df.show(5)

orders_df.printSchema()

print("Total Records :", orders_df.count())

In [0]:
null_df = orders_df.filter(
    col("order_id").isNull() |
    col("customer_id").isNull() |
    col("product_id").isNull() |
    col("quantity").isNull() |
    col("order_date").isNull()
)

print("Null Records :", null_df.count())

null_df.show()

In [0]:
valid_df = orders_df.filter(
    col("order_id").isNotNull() &
    col("customer_id").isNotNull() &
    col("product_id").isNotNull() &
    col("quantity").isNotNull() &
    col("order_date").isNotNull()
)

In [0]:
duplicate_df = (
    valid_df
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate Records :", duplicate_df.count())

duplicate_df.show()

In [0]:
silver_df = valid_df.dropDuplicates(["order_id"])

In [0]:
invalid_qty_df = silver_df.filter(col("quantity") <= 0)

print("Invalid Quantity Records :", invalid_qty_df.count())

invalid_qty_df.show()

In [0]:
silver_df = silver_df.filter(col("quantity") > 0)

In [0]:
future_orders = silver_df.filter(
    col("order_date") > current_date()
)

print("Future Orders :", future_orders.count())

future_orders.show()

In [0]:
silver_df = silver_df.filter(
    col("order_date") <= current_date()
)

In [0]:
silver_df = (
    silver_df
    .withColumn("load_timestamp", current_timestamp())
    .withColumn("data_source", lit("orders.csv"))
)

In [0]:
silver_df.show(5)

silver_df.printSchema()

print("Silver Records :", silver_df.count())

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.default.silver_orders")

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.silver_orders
LIMIT 10
""").show()